# Teks — TAPT: lanjut pretraining MLM di teks lomba, lalu fine-tune dari checkpoint baru
**Kapan dipakai:** bahasa/gaya teks lomba beda dari data pretraining model (berita lokal, ulasan
aplikasi, medsos, istilah domain hukum/medis), dan ada waktu GPU ± 10-30 menit sebelum
fine-tune. Task-Adaptive PreTraining (Gururangan dkk. 2020, "Don't Stop Pretraining") rata-rata
menaikkan skor 1-3 poin dan sering menang di Kaggle NLP.

**Cara:** semua teks train + test (+ teks tak berlabel lain bila ada) dipotong blok `BLOCK_SIZE`
token → 15% token ditutup `[MASK]` → model belajar menebaknya. TANPA label, jadi memakai teks test
tidak bocor label. Checkpoint disimpan ke `pretrained/tapt_<nama>`; di `finetune_oof` (atau
notebook fine-tune lain) cukup ganti `PRETRAINED` ke folder itu.

**Saat lomba yang diubah:** sel Konfigurasi. Patokan: 3-10 epoch, berhenti bila loss validasi
tidak turun lagi.

**Data demo:** teks SmSA train + test.

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling
from transformers import get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TEXT_FILES = [f"{DATA_FOLDER}/smsa_train.csv", f"{DATA_FOLDER}/smsa_test.csv"]   # GANTI
TEXT_COL = "text"                # GANTI saat lomba
PRETRAINED = "../../../pretrained/indobert"           # GANTI: model yang nanti di-fine-tune
# PRETRAINED = "../../../pretrained/indobertweet"     # medsos
# PRETRAINED = "../../../pretrained/xlmr_base"        # campur bahasa
SAVE_FOLDER = "../../../pretrained/tapt_indobert"     # GANTI: lalu pakai di finetune_oof
BLOCK_SIZE = 128                 # token per blok (samakan kira-kira dengan MAX_LENGTH fine-tune)
MASK_PROB = 0.15                 # bagian token yang ditutup
EPOCHS = 5
BATCH_SIZE = 32                  # blok panjang sama, jadi batch besar aman, 16 bila 512 token
LEARNING_RATE = 5e-5
WARMUP = 0.06
VAL_SHARE = 0.05                 # blok untuk mengukur loss validasi
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

## Kumpulkan teks
Teks kembar dibuang (tidak menambah informasi, hanya memperlama).

In [ ]:
texts = []
for path in TEXT_FILES:
    frame = pd.read_csv(path)
    for text in frame[TEXT_COL].fillna("").astype(str):
        cleaned = " ".join(html.unescape(text).split())
        if cleaned:
            texts.append(cleaned)
before = len(texts)
texts = list(dict.fromkeys(texts))
print(f"teks: {before} | unik: {len(texts)}")

## Potong jadi blok
Semua teks disambung (dipisah token [SEP]) lalu dipotong per `BLOCK_SIZE`; tiap blok diberi
token awal/akhir model. Sisa ekor yang kurang dari satu blok dibuang.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)
token_lists = tokenizer(texts, add_special_tokens=False)["input_ids"]
stream = []
for ids in token_lists:
    stream.extend(ids + [tokenizer.sep_token_id])
inner = BLOCK_SIZE - 2
blocks = []
for start in range(0, len(stream) - inner + 1, inner):
    piece = [tokenizer.cls_token_id] + stream[start:start + inner] + [tokenizer.sep_token_id]
    blocks.append({"input_ids": piece})
rng = np.random.default_rng(SEED)
order = rng.permutation(len(blocks))
n_val = max(1, int(len(blocks) * VAL_SHARE))
val_blocks, fit_blocks = order[:n_val], order[n_val:]
print(f"token: {len(stream)} | blok: {len(blocks)} | latih {len(fit_blocks)} validasi {n_val}")

## Model dan penutup token
`DataCollatorForLanguageModeling` memilih 15% token: 80% jadi [MASK], 10% kata acak, 10% tetap.
Loss hanya dihitung di token terpilih.

In [ ]:
torch.manual_seed(SEED)
model = AutoModelForMaskedLM.from_pretrained(PRETRAINED, local_files_only=True)
model = model.to(DEVICE)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=MASK_PROB)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
steps = int(np.ceil(len(fit_blocks) / BATCH_SIZE)) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)


def masked_loss(chosen):
    picked = []
    for number in chosen:
        picked.append(blocks[number])
    batch = collator(picked)
    batch = batch.to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        return model(**batch).loss


print("langkah per epoch:", int(np.ceil(len(fit_blocks) / BATCH_SIZE)))

In [ ]:
def val_loss():
    model.eval()
    torch.manual_seed(SEED)            # penutup token validasi sama tiap epoch
    total = []
    with torch.no_grad():
        for start in range(0, len(val_blocks), BATCH_SIZE):
            total.append(masked_loss(val_blocks[start:start + BATCH_SIZE]).item())
    model.train()
    return float(np.mean(total))


def train_epoch():
    model.train()
    shuffled = rng.permutation(fit_blocks)
    for start in range(0, len(shuffled), BATCH_SIZE):
        loss = masked_loss(shuffled[start:start + BATCH_SIZE])
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()


print(f"loss validasi sebelum TAPT: {val_loss():.4f}")

## Latih dan simpan checkpoint terbaik
Perplexity = exp(loss). Disimpan setiap kali loss validasi turun, jadi aman dihentikan di tengah.

In [ ]:
os.makedirs(SAVE_FOLDER, exist_ok=True)
best = np.inf
began = time.time()
for epoch in range(EPOCHS):
    train_epoch()
    value = val_loss()
    status = "-"
    if value < best:
        best = value
        model.save_pretrained(SAVE_FOLDER)
        tokenizer.save_pretrained(SAVE_FOLDER)
        status = "disimpan"
    print(f"epoch {epoch + 1}: loss val {value:.4f} | perplexity {np.exp(value):.2f} | "
          f"{time.time() - began:.0f} dtk | {status}")
print("checkpoint:", os.path.abspath(SAVE_FOLDER))

## Pakai checkpoint
Di `finetune_oof` / `finetune_multilabel` / `finetune_regresi` / `ner_bert`:
`PRETRAINED = "../../../pretrained/tapt_indobert"` dan `MODEL_NAME` baru (mis. `ft_tapt`), lalu
bandingkan skor OOF di `percobaan.csv` dengan versi tanpa TAPT. Peringatan "newly initialized"
untuk kepala klasifikasi itu normal.

In [ ]:
print(sorted(os.listdir(SAVE_FOLDER)))